# Strategy Template — Risk-Adjusted Cross-Sectional Alpha

Reproducible walk-forward research over `mart_signals_daily` using
`ccquant.strategy` (`cs_mom_oi_regime` by default).

**Pipeline:** PIT features → weekly L/S portfolio → costs → purged walk-forward → scale gates.

**Prereqs**
```bash
uv run ccquant sync all
uv run ccquant research run --strategy cs_mom_oi_regime
```

Contract: [`documentation/strategy_research.md`](../documentation/strategy_research.md).

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from dotenv import load_dotenv
from IPython.display import display

from ccquant.forecasting import load_signals_panel
from ccquant.strategy import run_strategy_detailed
from ccquant.strategy.spec import default_strategy_config_path, load_strategy_config

warnings.filterwarnings("ignore", category=FutureWarning)

_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
load_dotenv(_root / ".env", override=True)

DB = Path(os.environ.get("CCQUANT_DB", _root / "data" / "ccquant.duckdb")).resolve()
STRATEGY = "cs_mom_oi_regime"
CFG_PATH = default_strategy_config_path(STRATEGY)
print(f"DB: {DB}  exists={DB.exists()}")
print(f"Config: {CFG_PATH}")

## 1. Load panel + run strategy

Falls back to a synthetic panel when the DB / mart is missing so the notebook still runs top-to-bottom.

In [ ]:
import math
from datetime import date, timedelta

cfg = load_strategy_config(CFG_PATH)
panel: pl.DataFrame | None = None
source = "synthetic"

if DB.exists():
    try:
        panel = load_signals_panel(DB)
        if panel.height > 0:
            source = "mart_signals_daily"
    except Exception as exc:  # noqa: BLE001 — notebook degrade path
        print(f"load_signals_panel failed: {exc}")
        panel = None

if panel is None or panel.height == 0:

    def _synth(n_days: int = 500, symbols: tuple[str, ...] = ("AAA", "BBB", "CCC", "DDD", "EEE", "BTC")) -> pl.DataFrame:
        rows: list[dict[str, object]] = []
        start = date(2020, 1, 1)
        dates = [start + timedelta(days=i) for i in range(n_days)]
        for si, sym in enumerate(symbols):
            price = 100.0 * (1.0 + 0.1 * si)
            mu = 0.001 * (si + 1)
            for di, d in enumerate(dates):
                price *= 1.0 + mu + 0.01 * math.sin(di / 7.0 + si)
                rows.append(
                    {
                        "symbol": sym,
                        "date": d,
                        "open": price,
                        "high": price * 1.01,
                        "low": price * 0.99,
                        "close": price,
                        "volume": 1_000_000.0 / price,
                        "total_open_interest_usd": 5_000_000.0 * (1 + 0.05 * si),
                        "m2sl": 15_000.0 + di * 2.0,
                        "walcl": 8_000.0 + di * 1.0,
                        "dgs10": 3.0,
                        "t10yie": 2.0,
                    }
                )
        return pl.DataFrame(rows).with_columns(pl.col("date").cast(pl.Date))

    panel = _synth()
    print("Using synthetic panel (no mart available)")
else:
    print(f"Loaded {panel.height:,} rows · {panel['symbol'].n_unique()} symbols from {source}")

run = run_strategy_detailed(panel=panel, config=cfg, write_dir=_root / "data" / "research")
report = run.report
daily = run.daily
print(f"strategy={report.strategy_name} hash={report.config_hash} passed={report.passed}")
print(f"data_max_date={report.data_max_date} capacity_usd={report.capacity_usd:,.0f}")
if report.gate_reasons:
    for r in report.gate_reasons:
        print(f"  gate: {r}")
display(pl.DataFrame({"metric": list(report.oos_metrics.keys()), "value": list(report.oos_metrics.values())}))

## 2. Equity curve & drawdown

In [ ]:
eq = daily.select(["date", "equity_net", "equity_gross", "net_ret"]).drop_nulls()
fig = go.Figure()
fig.add_trace(go.Scatter(x=eq["date"], y=eq["equity_net"], name="equity_net", line=dict(width=2)))
fig.add_trace(go.Scatter(x=eq["date"], y=eq["equity_gross"], name="equity_gross", line=dict(width=1, dash="dot")))
fig.update_layout(title=f"{report.strategy_name} — equity (net vs gross)", template="plotly_white", height=420)
fig.show()

dd = eq.with_columns(
    (pl.col("equity_net") / pl.col("equity_net").cum_max() - 1.0).alias("drawdown")
)
fig_dd = px.area(dd.to_pandas(), x="date", y="drawdown", title="Net drawdown")
fig_dd.update_layout(template="plotly_white", height=320)
fig_dd.show()

## 3. Turnover, costs, regime context

In [ ]:
cost_cols = [c for c in ("turnover", "total_cost", "fee_cost", "slippage") if c in daily.columns]
if cost_cols:
    fig_t = px.line(
        daily.select(["date", *cost_cols]).to_pandas(),
        x="date",
        y=cost_cols,
        title="Turnover & costs",
    )
    fig_t.update_layout(template="plotly_white", height=360)
    fig_t.show()

print("OOS fold count:", len(report.fold_metrics))
if report.fold_metrics:
    fold_df = pl.DataFrame(list(report.fold_metrics))
    display(fold_df.select([c for c in ("fold", "net_sharpe", "ir_ew", "max_drawdown", "avg_turnover", "n_days") if c in fold_df.columns]))

## 4. Scale-gate verdict

Pass requires OOS net Sharpe > 0, IR(EW) > 0, and capacity ≥ target notional (see YAML gates).

In [ ]:
verdict = "PASSED" if report.passed else "FAILED"
print(f"Scale gates: {verdict}")
print(f"net_sharpe={report.oos_metrics.get('net_sharpe')}")
print(f"ir_ew={report.oos_metrics.get('ir_ew')}")
print(f"capacity_usd={report.capacity_usd:,.0f} vs target={report.target_notional_usd:,.0f}")
print("Report JSON under data/research/ (gitignored).")